# SLT Grokking Experiment — Colab Runner

**Workflow per session:**
1. Run **Section 0** (mount Drive + clone GitHub + symlink results). Always do this first.
2. Run **Section 1** (install deps). Always do this once per session.
3. Run the relevant section: Training / Calibration / LLC estimation / Figures.

Code comes from GitHub. Checkpoints and results persist on Drive across disconnects.

Repo: https://github.com/makataomu/slt-diplomka

## Section 0: Setup (run every session)

In [ ]:
# ── Mount Drive ───────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil, subprocess

REPO_URL     = 'https://github.com/makataomu/slt-diplomka'
REPO_DIR     = '/content/slt'
DRIVE_PERSIST = '/content/drive/MyDrive/slt_persist'  # all persistent data lives here

# ── Create Drive persistence folders ──────────────────────────────────────────
for d in ['results/checkpoints', 'results/metrics', 'results/figures']:
    os.makedirs(f'{DRIVE_PERSIST}/{d}', exist_ok=True)

# ── Clone or pull code from GitHub ───────────────────────────────────────────
if os.path.exists(f'{REPO_DIR}/.git'):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
    print('Pulled latest from GitHub')
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Cloned from GitHub')

# ── Symlink results/ -> Drive (checkpoints/metrics/figures persist) ───────────
results_link = f'{REPO_DIR}/results'
if os.path.islink(results_link):
    os.unlink(results_link)
elif os.path.isdir(results_link):
    shutil.rmtree(results_link)
os.symlink(f'{DRIVE_PERSIST}/results', results_link)
print(f'Symlinked results/ -> {DRIVE_PERSIST}/results')

# ── Restore llc_calibration.yaml from Drive (if previously calibrated) ────────
calib_drive = f'{DRIVE_PERSIST}/llc_calibration.yaml'
calib_local = f'{REPO_DIR}/configs/llc_calibration.yaml'
if os.path.exists(calib_drive):
    shutil.copy(calib_drive, calib_local)
    print('Restored llc_calibration.yaml from Drive')
else:
    print('No calibration config on Drive yet (will use defaults until calibration is run)')

# ── Set working directory and Python path ────────────────────────────────────
os.chdir(REPO_DIR)
if f'{REPO_DIR}/src' not in sys.path:
    sys.path.insert(0, f'{REPO_DIR}/src')

print(f'\nReady. Working dir: {os.getcwd()}')
print(f'Persistent storage: {DRIVE_PERSIST}')

## Section 1: Install dependencies (run once per Colab session)

In [ ]:
%pip install -q transformer_lens devinterp zarr==3.1.6

import importlib.metadata, torch
dv = importlib.metadata.version('devinterp')
print(f'devinterp {dv}  |  torch {torch.__version__}  |  device: {"cuda" if torch.cuda.is_available() else "cpu"}')
print('LLC estimation: custom SGLD loop (devinterp.optim.SGLD) — bypasses v2 API')

## Section 2: Training

Run one `(ratio, seed)` pair. Set `RESUME = True` after a disconnect to continue from the last checkpoint.

In [ ]:
# ── CONFIGURE ────────────────────────────────────────────────────────────────
RATIO  = 0.50   # one of [0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70]
SEED   = 0      # 0, 1, or 2
RESUME = False  # set True to resume from last checkpoint
# ─────────────────────────────────────────────────────────────────────────────

resume_flag = '--resume' if RESUME else ''
!python src/train.py --ratio {RATIO} --seed {SEED} {resume_flag}

## Section 3: LLC Calibration

Run **once** after the ratio=0.50 seed=0 training run is done.
Inspect the chain traces, fill in the calibrated values, then run the save cell.
Good calibration: chains fluctuate in a stable band (not diverging, not flatlined).

In [ ]:
# Run calibration (uses final checkpoint of ratio=0.50 seed=0)
!python src/llc_estimation.py --ratio 0.50 --seed 0 --calibrate

In [ ]:
# Plot chain traces
import numpy as np
import matplotlib.pyplot as plt

traces = np.load('results/metrics/calibration_traces.npy')
print(f'Shape: {traces.shape}  (chains × draws)')

fig, ax = plt.subplots(figsize=(10, 4))
for i, chain in enumerate(traces):
    ax.plot(chain, lw=0.8, alpha=0.7, label=f'Chain {i}')
ax.set(xlabel='Draw', ylabel='Loss (SGLD)',
       title='SGLD calibration traces — chains should mix, not diverge or flatline')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print('Chains diverge  → lower epsilon (e.g. 1e-5)')
print('Chains flatline → raise epsilon or num_draws')

In [ ]:
# Save calibrated hyperparams (edit values first, then run)
import yaml
from pathlib import Path

# ── FILL IN after inspecting traces above ─────────────────────────────────────
# nbeta starting point: default_nbeta(256) = 256/log(256) ≈ 46.2
# Adjust based on chain trace health (stable band = good)
CALIBRATED = dict(
    calibrated           = True,
    epsilon              = 1e-4,
    nbeta                = 46.2,   # default_nbeta(256); adjust if chains look bad
    gamma                = 10.0,
    num_chains           = 8,
    num_draws            = 500,
    num_burnin_steps     = 100,
    calibration_checkpoint = 'results/checkpoints/ratio_0.50/seed_0/epoch_06000.pt',
    calibration_date     = '2026-XX-XX',
    calibration_notes    = '',
)
# ─────────────────────────────────────────────────────────────────────────────

yaml_str = yaml.dump(CALIBRATED, default_flow_style=False)
Path('configs/llc_calibration.yaml').write_text(yaml_str)

# Back up to Drive so it survives re-clones
import shutil
shutil.copy('configs/llc_calibration.yaml', f'{DRIVE_PERSIST}/llc_calibration.yaml')

print('Saved configs/llc_calibration.yaml and backed up to Drive')
print(yaml_str)

## Section 4: LLC Estimation

Run after training + calibration. Processes all 100 checkpoints for one (ratio, seed).
Estimate: ~2–4 min per checkpoint × 100 = **3–7 hours per run**.

**Ask Tair before starting** — this is >2h compute.

In [ ]:
# ── CONFIGURE ────────────────────────────────────────────────────────────────
RATIO = 0.50
SEED  = 0
# ─────────────────────────────────────────────────────────────────────────────

import yaml
cfg = yaml.safe_load(open('configs/llc_calibration.yaml'))
if not cfg.get('calibrated'):
    print('ERROR: run Section 3 (calibration) first.')
else:
    print(f'epsilon={cfg["epsilon"]}  nbeta={cfg["nbeta"]}  gamma={cfg["gamma"]}')
    !python src/llc_estimation.py --ratio {RATIO} --seed {SEED}

## Section 5: Generate Figures

In [ ]:
!python src/plotting.py

from pathlib import Path
for f in sorted(Path('results/figures').glob('*.pdf')):
    print(f'  {f.name}  ({f.stat().st_size:,} bytes)')

## Section 6: Full training sweep (advanced)

Runs all 7 ratios × 3 seeds. Only if you have ~12h uninterrupted.
Training only — LLC estimation needs separate approval.

In [ ]:
RATIOS = [0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70]
SEEDS  = [0, 1, 2]

for ratio in RATIOS:
    for seed in SEEDS:
        print(f'\n=== ratio={ratio}  seed={seed} ===')
        !python src/train.py --ratio {ratio} --seed {seed} --epochs 6000 --checkpoint_every 60 --resume

print('\nAll training runs complete.')

## Section 7: Diagnostics — what's been computed so far

In [ ]:
from pathlib import Path
import pandas as pd

print('=== Checkpoints ===')
for ratio_dir in sorted(Path('results/checkpoints').glob('ratio_*')):
    for seed_dir in sorted(ratio_dir.glob('seed_*')):
        ckpts = sorted(seed_dir.glob('epoch_*.pt'))
        if ckpts:
            last = int(ckpts[-1].stem.split('_')[1])
            print(f'  {ratio_dir.name}/{seed_dir.name}: {len(ckpts)} checkpoints, last epoch={last}')

print('\n=== Training metrics ===')
for f in sorted(Path('results/metrics').glob('ratio_*.csv')):
    if '_llc' not in f.name:
        df = pd.read_csv(f)
        print(f'  {f.name}: {len(df)} rows, max_epoch={df["epoch"].max()}')

print('\n=== LLC metrics ===')
for f in sorted(Path('results/metrics').glob('*_llc.csv')):
    df = pd.read_csv(f)
    print(f'  {f.name}: {len(df)} checkpoints estimated')